# NOVA Auto-Learning — Google Colab (Temporary Runner)

Readiness-first manual runner. GitHub `main` remains the source of truth. The notebook stops until the reviewed Learning Engine, trainer, and versioned Exness data are present on `main`.

**Security:** add credentials only in Colab Secrets (key icon). Never put token values in cells, outputs, notebook metadata, GitHub, or chat. **Safety:** historical research candidate only; no broker writes, orders, auto-promotion, or live-trading activation.


## Colab Secrets
Add `GITHUB_READ_TOKEN` (fine-grained, Contents Read-only for `vutting895/VOOM_SYSTEM-NOVA-AI`). Optional `NOVA_DATASET_ID` defaults to `exness_xauusdm_m15`. Add `HF_TOKEN` and `HF_MODEL_REPO_ID` only when a gated candidate upload is deliberately approved. Keep the notebook Private.


In [ ]:
from google.colab import userdata
def read_secret(name, required=False):
    try: value = userdata.get(name)
    except Exception: value = None
    value = value.strip() if isinstance(value, str) else None
    if required and not value: raise RuntimeError(f'Missing Colab Secret: {name}')
    return value or None
GITHUB_READ_TOKEN = read_secret('GITHUB_READ_TOKEN', True)
NOVA_DATASET_ID = read_secret('NOVA_DATASET_ID') or 'exness_xauusdm_m15'
EXTRA_SENSITIVE = []
print('GitHub read credential: configured; value hidden')
print('Hugging Face credentials are not read unless upload is explicitly enabled.')


In [ ]:
import os, base64, shutil, subprocess
from pathlib import Path
REPOSITORY = 'https://github.com/vutting895/VOOM_SYSTEM-NOVA-AI.git'
WORKDIR = Path('/content/VOOM_SYSTEM-NOVA-AI')
def run_safe(command, cwd=None, env=None, timeout=3600):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        detail = (result.stderr or result.stdout or 'No command output')[-1800:]
        for secret in [GITHUB_READ_TOKEN] + EXTRA_SENSITIVE:
            if secret: detail = detail.replace(secret, '[REDACTED]')
        raise RuntimeError(f'Command failed with exit code {result.returncode}:\\n{detail}')
    return result.stdout
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
auth = base64.b64encode(('x-access-token:' + GITHUB_READ_TOKEN).encode()).decode('ascii')
env['GIT_CONFIG_COUNT']='1'; env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']='AUTHORIZATION: basic ' + auth
try: run_safe(['git','clone','--depth','1','--branch','main',REPOSITORY,str(WORKDIR)],env=env)
finally:
    for key in ('GIT_CONFIG_COUNT','GIT_CONFIG_KEY_0','GIT_CONFIG_VALUE_0'): env.pop(key,None)
    auth=None; del env; GITHUB_READ_TOKEN=None
MAIN_SHA = run_safe(['git','rev-parse','HEAD'],cwd=WORKDIR).strip()
print('Cloned reviewed main; commit:', MAIN_SHA)


In [ ]:
import json, hashlib, subprocess
required = {
 'Learning Engine loader': WORKDIR/'backend/learning/data_sources.py',
 'Candidate trainer': WORKDIR/'backend/learning/train_price_model.py',
 'Training requirements': WORKDIR/'backend/learning/requirements-training.txt',
 'Exness dataset manifest': WORKDIR/'datasets/exness/demo/xauusdm_m15/dataset_manifest.json',
}
missing = [name for name,path in required.items() if not path.is_file()]
for name,path in required.items(): print(('FOUND' if path.is_file() else 'MISSING') + ' — ' + name)
if missing: raise RuntimeError('Blocked until PR #197 and PR #237 dependencies are reviewed, merged, and verified on main: '+', '.join(missing))
manifest_path=required['Exness dataset manifest']; manifest=json.loads(manifest_path.read_text())
if manifest.get('benchmark_eligibility')!='candidate_only': raise RuntimeError('Refusing non-candidate dataset; frozen Wave3 benchmark must not be replaced.')
if manifest.get('account_mode')!='DEMO' or manifest.get('orders_submitted') is not False or manifest.get('live_trading_enabled') is not False: raise RuntimeError('Exness source is not verified Demo/read-only data.')
if subprocess.run(['git','lfs','version'],capture_output=True).returncode: raise RuntimeError('Git LFS is required for historical CSV retrieval.')
run_safe(['git','lfs','pull'],cwd=WORKDIR)
def verify_file(name,size,sha):
    p=manifest_path.parent/name
    if not p.is_file() or p.stat().st_size!=int(size) or hashlib.sha256(p.read_bytes()).hexdigest()!=sha: raise RuntimeError('Dataset file size/SHA-256 verification failed: '+name)
    print('Verified size and SHA-256:',name)
verify_file(manifest['csv_file'],manifest['csv_size_bytes'],manifest['csv_sha256'])
verify_file(manifest['metadata_file'],manifest['source_metadata_size_bytes'],manifest['metadata_source_sha256'])
print('Dataset integrity passed; candidate_only; final OOS untouched.')


## Install and run
The dependency and checksum gate must pass before installing the trainer. The two flags below are deliberately disabled. Do not schedule this notebook or change the flags until exact-head CI is inspectably green and all dependencies are reviewed.


In [ ]:
run_safe(['python','-m','pip','install','-r',str(required['Training requirements'])],cwd=WORKDIR)
print('Training dependencies installed.')


In [ ]:
RUN_TRAINING = False
UPLOAD_TO_HF = False
if not RUN_TRAINING:
    print('Training skipped: RUN_TRAINING is False.')
else:
    train_env=os.environ.copy(); hf_token=None; model_repo_id=None
    if UPLOAD_TO_HF:
        hf_token=read_secret('HF_TOKEN',True); model_repo_id=read_secret('HF_MODEL_REPO_ID',True)
        EXTRA_SENSITIVE.extend([hf_token,model_repo_id])
        train_env['HF_TOKEN']=hf_token; train_env['HF_MODEL_REPO_ID']=model_repo_id
    cmd=['python','-m','backend.learning.train_price_model','--repo-root',str(WORKDIR),'--dataset-id',NOVA_DATASET_ID,'--output-root','artifacts/learning-candidates']
    if UPLOAD_TO_HF: cmd.append('--upload-to-hf')
    try:
        result=run_safe(cmd,cwd=WORKDIR,env=train_env); print(result[-3500:])
    finally:
        train_env.pop('HF_TOKEN',None); train_env.pop('HF_MODEL_REPO_ID',None)
        EXTRA_SENSITIVE.clear(); hf_token=None; model_repo_id=None; del train_env
        print('Temporary upload credential references cleared.')


## Handoff and safety
Preserve `MAIN_SHA`, dataset SHA-256, manifest, metrics, and run artifacts. The trainer uses chronological splits and keeps the final holdout unscored. Uploads, when explicitly enabled and gated, go only to immutable private Hugging Face folders; there is no `latest` alias, promotion, inference activation, broker write, or live trading. Return to Kaggle for scheduled training after the mobile Editor issue and all CI/dependency gates are resolved.
